# Phase 5: Business Insights

Turn model outputs into actionable interventions:
1. Risk scoring (Low / Medium / High)
2. Uplift-based customer segmentation (Persuadable / Sure Thing / Sleeping Dog / Lost Cause)
3. Per-customer churn explanation reports
4. ROI analysis of targeted intervention

**Prerequisites**: `02_feature_engineering.ipynb`, `03_modeling.ipynb`, `04_causal_inference.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data & Model

In [ ]:
import os

# churn_causal.csv is produced by notebook 04 (includes uplift scores).
# Fall back to churn_features.csv if 04 hasn't been run yet —
# risk scoring and SHAP still work; segmentation will be skipped.
causal_path   = '../data/churn_causal.csv'
features_path = '../data/churn_features.csv'

if os.path.exists(causal_path):
    data = pd.read_csv(causal_path)
    HAS_UPLIFT = 'uplift_X_T_MonthToMonth' in data.columns
    print(f'Loaded {causal_path} — uplift columns present: {HAS_UPLIFT}')
else:
    data = pd.read_csv(features_path)
    HAS_UPLIFT = False
    print(f'churn_causal.csv not found — loaded {features_path}. '
          'Run 04_causal_inference.ipynb for uplift-based segmentation.')

calibrated_model = joblib.load('../models/calibrated_model.pkl')
threshold        = joblib.load('../models/threshold.pkl')
feature_cols     = joblib.load('../models/feature_columns.pkl')

X = data[feature_cols]
print(f'Dataset: {data.shape}, threshold: {threshold}')

## 2. Risk Scoring

In [ ]:
LOW_THRESHOLD  = 0.3
HIGH_THRESHOLD = 0.6

proba = calibrated_model.predict_proba(X)[:, 1]

risk_level = pd.cut(
    proba,
    bins=[-0.001, LOW_THRESHOLD, HIGH_THRESHOLD, 1.001],
    labels=['Low', 'Medium', 'High']
)

data['Churn_Probability'] = proba
data['Risk_Level'] = risk_level

print('Risk level distribution:')
print(data['Risk_Level'].value_counts(normalize=True).round(3))

In [ ]:
risk_actions = {
    'Low':    'No action needed — monitor quarterly.',
    'Medium': 'Proactive outreach: satisfaction survey + loyalty offer.',
    'High':   'Immediate intervention: contract upgrade offer + account manager call.',
}
data['Recommended_Action'] = data['Risk_Level'].map(risk_actions)
data[['Churn_Probability', 'Risk_Level', 'Recommended_Action']].head(10)

## 3. Per-Customer Explanation

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np

# Use the XGB component of the ensemble for SHAP explanations
X_train_raw = np.load('../data/X_train_raw.npy')
y_train_raw = np.load('../data/y_train_raw.npy')

xgb_explain = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
xgb_explain.fit(X_train_raw, y_train_raw)
explainer = shap.TreeExplainer(xgb_explain)

In [ ]:
def explain_customer(index, top_n=3):
    """Return top_n features driving churn risk for a single customer."""
    row = X.iloc[[index]]
    shap_vals = explainer.shap_values(row)[0]
    pairs = sorted(zip(feature_cols, shap_vals, row.values[0]),
                   key=lambda x: abs(x[1]), reverse=True)[:top_n]
    return pairs

def customer_report(index):
    proba_val = proba[index]
    risk = risk_level.iloc[index]
    top = explain_customer(index)
    reasons = []
    for feat, impact, val in top:
        direction = 'increases' if impact > 0 else 'decreases'
        reasons.append(f'  • {feat}={val:.2f} {direction} churn risk (SHAP={impact:+.3f})')
    return {
        'churn_probability': round(float(proba_val), 3),
        'risk_level': str(risk),
        'top_drivers': '\n'.join(reasons)
    }

# Demo: customer at index 10 (High risk)
report = customer_report(10)
print(f"Churn probability : {report['churn_probability']}")
print(f"Risk level        : {report['risk_level']}")
print(f"Top drivers:\n{report['top_drivers']}")

In [ ]:
# Waterfall plot for one high-risk customer
high_risk_idx = data[data['Risk_Level'] == 'High'].index[0]
shap_vals_single = explainer(X.iloc[[high_risk_idx]])
shap.plots.waterfall(shap_vals_single[0])

## 4. Uplift-Based Segmentation

Using the X-learner uplift from `T_MonthToMonth` (the most actionable treatment):

| Segment | Churn prob | Uplift | Strategy |
|---|---|---|---|
| **Persuadable** | High | High (positive) | Intervene — will respond |
| **Sure Thing** | Low | Low | Monitor only |
| **Sleeping Dog** | Low/High | Negative | Don't contact — may increase churn |
| **Lost Cause** | High | Low | Not worth the cost |

In [ ]:
if not HAS_UPLIFT:
    print('Uplift columns not found — run 04_causal_inference.ipynb first, then re-run this notebook for full segmentation.')
else:
    df_seg = data.copy()
    df_seg['churn_prob'] = proba
    df_seg['uplift'] = df_seg['uplift_X_T_MonthToMonth']

    high_churn = df_seg['churn_prob'] >= df_seg['churn_prob'].median()
    pos_uplift = df_seg['uplift'] > 0

    df_seg['segment'] = 'Sure Thing'
    df_seg.loc[ high_churn &  pos_uplift, 'segment'] = 'Persuadable'
    df_seg.loc[ high_churn & ~pos_uplift, 'segment'] = 'Lost Cause'
    df_seg.loc[~high_churn & ~pos_uplift, 'segment'] = 'Sleeping Dog'

    print('Segment counts:')
    print(df_seg['segment'].value_counts())
    print('\nMean churn prob by segment:')
    print(df_seg.groupby('segment')['churn_prob'].mean().round(3))

## 5. Segment Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Segment distribution
df_seg['segment'].value_counts().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Customer Segments')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=30)

# Churn probability by segment
df_seg.boxplot(column='churn_prob', by='segment', ax=axes[0, 1])
axes[0, 1].set_title('Churn Probability by Segment')
axes[0, 1].set_xlabel('')

# Uplift by segment
df_seg.boxplot(column='uplift', by='segment', ax=axes[1, 0])
axes[1, 0].set_title('Uplift by Segment')
axes[1, 0].set_xlabel('')

# Monthly charges by segment
df_seg.boxplot(column='MonthlyCharges', by='segment', ax=axes[1, 1])
axes[1, 1].set_title('Monthly Charges by Segment')
axes[1, 1].set_xlabel('')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Segment Action Recommendations

---

### Persuadable — *Highest Priority*
- Offer contract upgrade (month-to-month → annual) with 10% discount
- Proactive TechSupport outreach
- Targeted loyalty programme enrollment

### Lost Cause
- Do not invest in retention offers — low ROI
- Focus on offboarding experience to protect brand reputation
- Use as signal to review onboarding for similar profiles

### Sleeping Dog
- **Do not contact** — intervention may increase churn (negative uplift)
- Passive satisfaction monitoring only

### Sure Thing
- Low priority; standard communications only
- Potential upsell candidates

## 7. ROI Analysis

In [ ]:
CAC               = 350    # cost to acquire a replacement customer
INTERVENTION_COST = 35     # cost per retention offer (discount + ops)
DISCOUNT_RATE     = 0.10   # 10% monthly charge discount offered
AVG_RETENTION_MONTHS = 24  # expected extra tenure if retained

roi_rows = []
for seg, group in df_seg.groupby('segment'):
    n = len(group)
    avg_charge = group['MonthlyCharges'].mean()
    avg_uplift = group['uplift'].mean()

    # Expected churners prevented
    prevented = n * max(avg_uplift, 0)  # negative uplift → 0 prevention

    total_cost = n * INTERVENTION_COST
    discount_cost = n * avg_charge * DISCOUNT_RATE * AVG_RETENTION_MONTHS * max(avg_uplift, 0)
    cac_avoided = prevented * CAC
    net_value = cac_avoided - total_cost - discount_cost

    roi_rows.append({
        'segment': seg,
        'n_customers': n,
        'avg_monthly_charge': round(avg_charge, 2),
        'avg_uplift': round(avg_uplift, 2),
        'intervention_cost': round(total_cost, 2),
        'net_value': round(net_value, 2),
        'ROI_%': round(net_value / total_cost * 100, 1) if total_cost > 0 else 0,
    })

roi_df = pd.DataFrame(roi_rows).set_index('segment')
print(roi_df.to_string())

In [ ]:
persuadable = roi_df.loc['Persuadable']
print('\n>>> Targeting Persuadables only:')
print(f'  Customers targeted : {persuadable["n_customers"]}')
print(f'  Intervention cost  : ${persuadable["intervention_cost"]:,.0f}')
print(f'  Net value          : ${persuadable["net_value"]:,.0f}')
print(f'  ROI                : {persuadable["ROI_%"]}%')

prevented = persuadable['n_customers'] * max(persuadable['avg_uplift'], 0)
print(f'  CAC alternative    : {prevented:.0f} customers × ${CAC} CAC = ${prevented * CAC:,.0f} avoided')

## 8. Save Final Output

In [ ]:
df_seg.to_csv('../data/churn_risk_results.csv', index=False)
roi_df.to_csv('../data/roi_analysis.csv')
print('Saved: churn_risk_results.csv, roi_analysis.csv')